# SuperSimpleNet — Augmentation ablation runner

Runs **every** augmentation config in `configs/*.json` (or a chosen subset), one
Drive folder per config. **Stop/resume-safe**: on restart it skips any config
that already finished (a `DONE.txt` marker), so you never lose completed work.

Workflow:
1. Run **Setup** once (mount Drive, clone your branch, install deps).
2. Run **Copy dataset to local** once (fast I/O; re-points `DATA_PATH` locally).
3. (SUP mode only) Run **Dataset prep** once to seed anomalies into the train set.
4. Run the **Grid runner** — re-run it any time; it resumes where it left off.
5. Run **Summary** to collect all metrics into one CSV on Drive.


## 1) Setup — mount Drive, clone branch, install dependencies

In [2]:
import os
from google.colab import drive

drive.mount('/content/drive')

# ========================= CONFIGURE ME =========================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
# IMPORTANT: your augmentation branch (must carry aug_config.py / augment_ssn.py / configs/).
BRANCH_NAME  = 'augmentation'
REPO_PATH    = '/content/SuperSimpleNet'

DATA_PATH    = '/content/drive/MyDrive/Tesi/datasets/MVTec'
# Base Drive folder for the WHOLE ablation grid (one sub-folder per config is created here).
ABLATION_BASE = '/content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation'

CATEGORY = 'Nero_SEMISUPERVISED_dustValidationAndTrain'
MODE     = 'sup'          # 'sup' (supervised/mixed) or 'unsup'

# Shared training hyperparameters (identical across configs so only augmentation varies)
EPOCHS = 300
BATCH  = 4
# Multi-seed ablation (SK-RD4AD style): every config is trained once per seed,
# then metrics are aggregated as mean +/- std across seeds in the Summary cell.
SEEDS  = [0, 1, 42]
# ===============================================================

if not os.path.exists(REPO_PATH):
    print(f">>> Cloning branch '{BRANCH_NAME}'...")
    !git clone -b {BRANCH_NAME} {GIT_REPO_URL} {REPO_PATH}
else:
    print(f">>> Repo present. Checking out '{BRANCH_NAME}' and pulling...")
    os.chdir(REPO_PATH)
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

os.chdir(REPO_PATH)
os.makedirs(ABLATION_BASE, exist_ok=True)
print("Working dir:", os.getcwd())
print("Ablation base:", ABLATION_BASE)

!pip install tqdm anomalib==0.7
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118
!pip install wandb optuna
# ONNX deps (needed only if RUN_ONNX_EXPORT=True in the grid runner)
!pip install onnx onnxscript onnxruntime onnxconverter-common
# CRITICAL: keep NumPy < 2.0 -- anomalib 0.7 pulls imgaug, which uses np.sctypes
# (removed in NumPy 2.0). Pin it LAST so no earlier install can bump it back to 2.x.
# train.py runs as a subprocess and reads this on-disk numpy, so no restart is needed.
!pip install "numpy<2.0"


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
>>> Repo present. Checking out 'augmentation' and pulling...
Already on 'augmentation'
Your branch is up to date with 'origin/augmentation'.
From https://github.com/EmanuelePietroCometti/SuperSimpleNet
 * branch            augmentation -> FETCH_HEAD
Already up to date.
Working dir: /content/SuperSimpleNet
Ablation base: /content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
ERROR: Could not find a version that satisfies the requirement torch==2.1.0+cu118 (from versions: 2.2.0, 2.2.0+cu118, 2.2.1, 2.2.1+cu118, 2.2.2, 2.2.2+cu118, 2.3.0, 2.3.0+cu118, 2.3.1, 2.3.1+cu118, 2.4.0, 2.4.0+cu118, 2.4.1, 2.4.1+cu118, 2.5.0, 2.5.0+cu118, 2.5.1, 2.5.1+cu118, 2.6.0, 2.6.0+cu118, 2.7.0, 2.7.0+cu118, 2.7.1, 2.7.1+cu118, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0)
E

## 1b) Copy dataset to local storage (fast I/O)

Reading the test/train images over Google Drive (FUSE) is the main reason eval is
slow, and in `sup` mode the loader is stuck at 1 worker. This copies the category
to local `/content` storage once and re-points `DATA_PATH` there for the rest of
the notebook. Idempotent: skipped if the local copy already exists (per session).


In [2]:
import os, shutil, time

# Source on Drive (whatever Setup pointed DATA_PATH at) -> local destination.
DRIVE_DATA = DATA_PATH
LOCAL_DATA = '/content/mvtec_local'
src = os.path.join(DRIVE_DATA, CATEGORY)
dst = os.path.join(LOCAL_DATA, CATEGORY)

if not os.path.isdir(src):
    raise FileNotFoundError(f"Source category not found on Drive: {src}")

if os.path.isdir(dst) and os.listdir(dst):
    print(f">>> Local copy already present, skipping copy: {dst}")
else:
    os.makedirs(LOCAL_DATA, exist_ok=True)
    print(f">>> Copying {src}\n           -> {dst} ...")
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f">>> Copy done in {time.time() - t0:.1f}s")

# Point the pipeline at the local copy for all downstream cells.
DATA_PATH = LOCAL_DATA
print(">>> DATA_PATH now ->", DATA_PATH)


>>> Copying /content/drive/MyDrive/Tesi/datasets/MVTec/Nero_SEMISUPERVISED_dustValidationAndTrain
           -> /content/mvtec_local/Nero_SEMISUPERVISED_dustValidationAndTrain ...
>>> Copy done in 276.8s
>>> DATA_PATH now -> /content/mvtec_local


## 2) Dataset prep (SUP mode only — run once)

Seeds a few anomalous samples (with masks) into the train set, exactly like the
main notebook. Idempotent: safe to re-run. Skip entirely for `unsup` mode.

In [3]:
import os, glob, shutil

NUM_ANOMALIES_PER_DEFECT = 2

if MODE == 'sup':
    dataset_root = os.path.join(DATA_PATH, CATEGORY)
    test_dir  = os.path.join(dataset_root, 'test')
    gt_root   = os.path.join(dataset_root, 'ground_truth')
    train_dir = os.path.join(dataset_root, 'train')

    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Cannot find test folder: {test_dir}")

    defect_types = [d for d in os.listdir(test_dir)
                    if os.path.isdir(os.path.join(test_dir, d)) and d != 'good']
    print(f">>> SUP prep for '{CATEGORY}', defects: {defect_types}")

    for defect in defect_types:
        defect_test_dir = os.path.join(test_dir, defect)
        defect_gt_dir   = os.path.join(gt_root, defect)
        target_train_defect_dir = os.path.join(train_dir, defect)
        target_train_gt_dir     = os.path.join(train_dir, 'ground_truth', defect)
        os.makedirs(target_train_defect_dir, exist_ok=True)
        os.makedirs(target_train_gt_dir, exist_ok=True)

        images = sorted(glob.glob(os.path.join(defect_test_dir, "*.png")))
        for img_path in images[:NUM_ANOMALIES_PER_DEFECT]:
            stem = os.path.splitext(os.path.basename(img_path))[0]
            dest_img = os.path.join(target_train_defect_dir, os.path.basename(img_path))
            if not os.path.exists(dest_img):
                shutil.copy(img_path, dest_img)
            for mask_path in (os.path.join(defect_gt_dir, f"{stem}_mask.png"),
                              os.path.join(defect_gt_dir, f"{stem}.png")):
                if os.path.exists(mask_path):
                    dest_mask = os.path.join(target_train_gt_dir, os.path.basename(mask_path))
                    if not os.path.exists(dest_mask):
                        shutil.copy(mask_path, dest_mask)
                    break
    print(">>> SUP dataset prep done.")
else:
    print(">>> MODE is not 'sup' -> skipping dataset prep.")


>>> SUP prep for 'Nero_SEMISUPERVISED_dustValidationAndTrain', defects: ['grappola', 'paglia', 'nodo_r40', 'nodo', 'splycer']
>>> SUP dataset prep done.


## 3) Grid runner (resume-safe, multi-seed)

Runs each config **once per seed** in `SEEDS`, with identical hyperparameters,
saving each run to its own Drive folder
`ABLATION_BASE/<CATEGORY>__<config>__seed<seed>/`. A `(config, seed)` pair that
already has a `DONE.txt` is **skipped**, so you can stop and re-run this cell
freely — completed seeds are never recomputed.

To run a subset, set `CONFIG_FILES` manually. To change the seed set, edit
`SEEDS` in the Setup cell.

In [4]:
import os, sys, glob, json, shutil, subprocess

os.chdir(REPO_PATH)


def run_streamed(cmd, log_path=None):
    """Run a command and STREAM its output into the notebook cell.

    subprocess output is NOT shown in Colab by default (it goes to the kernel's
    real stdout, not the cell), which hides tracebacks. We pipe it and re-print
    every line via print(), and optionally tee it to a log file on Drive.
    """
    print(">>>", " ".join(str(c) for c in cmd), flush=True)
    logf = open(log_path, "w", encoding="utf-8") if log_path else None
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
            if logf:
                logf.write(line)
    finally:
        proc.stdout.close()
        rc = proc.wait()
        if logf:
            logf.close()
    return rc


# ===================== CONFIGS TO RUN (edit freely) =====================
# Explicit list, grouped as in configs/EXPERIMENTS.md. Comment out any row you
# don't want, or reorder. To instead run EVERY json present automatically, use
# the glob line at the bottom.
CONFIG_FILES = [
    "configs/aug_off.json",
    "configs/aug_legacy.json",
    "configs/oat_affine.json",
    "configs/oat_blur.json",
    "configs/oat_brightness_contrast.json",
    "configs/oat_dynamic_crop.json",
    "configs/oat_equalize.json",
    "configs/oat_grayscale.json",
    "configs/oat_hflip.json",
    "configs/oat_hue.json",
    "configs/oat_speckle_high.json",
    "configs/oat_speckle_low.json",
    "configs/oat_vflip.json",
    "configs/combined_geometric.json",
    "configs/combined_candidate.json",
]
# Alternative: run every json present, no manual list:
# CONFIG_FILES = sorted(glob.glob('configs/*.json'))

# Keep only the ones that actually exist on disk (guards against typos/renames).
CONFIG_FILES = [c for c in CONFIG_FILES if os.path.exists(c)]
# =======================================================================

RUN_ONNX_EXPORT = True   # also export ONNX (needs the onnx deps from the setup cell)

n_runs = len(CONFIG_FILES) * len(SEEDS)
print(f"{len(CONFIG_FILES)} configs x {len(SEEDS)} seeds = {n_runs} runs queued")
print(f"seeds: {SEEDS}")
for c in CONFIG_FILES:
    print("   -", os.path.basename(c))


def _find_weights(root):
    hits = glob.glob(os.path.join(root, "**", "weights.pt"), recursive=True)
    return sorted(hits)[0] if hits else None


# Nested loop: for each config, run every seed as an INDEPENDENT run.
# Each (config, seed) gets its own run_dir + setup_name + DONE.txt, so the
# resume logic works per seed and per-seed metrics never overwrite each other.
for cfg_path in CONFIG_FILES:
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]

    for seed in SEEDS:
        run_dir     = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}__seed{seed}")
        done_marker = os.path.join(run_dir, "DONE.txt")
        setup_name  = f"ssn_{cfg_name}_seed{seed}"

        if os.path.exists(done_marker):
            print(f"[SKIP] {cfg_name} seed={seed} already completed")
            continue

        os.makedirs(run_dir, exist_ok=True)
        print("\n" + "=" * 64)
        print(f"[RUN ] {cfg_name}  seed={seed}  ->  {run_dir}")
        print("=" * 64)

        # --- 1) Train (skip if a checkpoint already exists but the run isn't DONE:
        #        e.g. it crashed during eval/export -> resume at eval, don't retrain). ---
        weights = _find_weights(run_dir)
        if weights:
            print(f"[TRAIN] existing checkpoint found, skipping training -> {weights}")
        else:
            rc = run_streamed([
                "python", "train.py",
                "--dataset", "mvtec",
                "--category", CATEGORY,
                "--mode", MODE,
                "--data_path", DATA_PATH,
                "--datasets_folder", DATA_PATH,
                "--results_save_path", run_dir,
                "--setup_name", setup_name,
                "--num_workers", "1",
                "--backbone", "wide_resnet50_2",
                "--layers", "layer1", "layer2", "layer3",
                "--image_size", "256", "256",
                "--epochs", str(EPOCHS),
                "--batch", str(BATCH),
                "--perlin_thr", "0.34628",
                "--noise_std", "0.10580",
                "--seg_lr", "2.087e-5",
                "--dec_lr", "7.798e-4",
                "--adapt_lr", "0.0001",
                "--patch_size", "5",
                "--gamma", "0.52232",
                "--eval_step_size", "20",
                "--seed", str(seed),
                "--aug_config", cfg_path,
            ], log_path=os.path.join(run_dir, "train_log.txt"))
            if rc != 0:
                print(f"[FAIL] {cfg_name} seed={seed} training (exit {rc}) -- see train_log.txt above; will retry next run")
                continue
            weights = _find_weights(run_dir)

        if not weights:
            print(f"[WARN] {cfg_name} seed={seed}: training finished but no weights.pt found; skipping eval")
            continue

        # --- 2) Eval: this is what writes the <weights>.calib.json sidecar (train.py
        #        alone does NOT). Without it the ONNX export has no calibration. ---
        print(f"[EVAL] {cfg_name} seed={seed} -> writing .calib.json + eval report")
        ev_rc = run_streamed([
            "python", "eval.py", weights,
            "--dataset", "mvtec",
            "--category", CATEGORY,
            "--datasets_folder", DATA_PATH,
            "--results_save_path", run_dir,
            "--backbone", "wide_resnet50_2",
            "--image_size", "256", "256",
            "--batch", str(BATCH),
            "--num_workers", "1",
            "--seed", str(seed),
            "--layers", "layer1", "layer2", "layer3",
            "--patch_size", "5",
        ], log_path=os.path.join(run_dir, "eval_log.txt"))
        calib = weights + ".calib.json"
        if ev_rc != 0 or not os.path.exists(calib):
            print(f"[FAIL] {cfg_name} seed={seed} eval/calib (exit {ev_rc}, calib={os.path.exists(calib)}) "
                  f"-- see eval_log.txt above; no DONE marker, will retry (training is preserved)")
            continue
        print(f"[calib] OK -> {calib}")

        # --- 3) Optional ONNX export, gathered next to the checkpoint. ---
        if RUN_ONNX_EXPORT:
            print(f"[ONNX] {cfg_name} seed={seed} -> exporting")
            run_streamed(["python", "export_onnx.py", weights],
                         log_path=os.path.join(run_dir, "export_log.txt"))
            onnx_dir = os.path.join(run_dir, "onnx")
            os.makedirs(onnx_dir, exist_ok=True)
            for onnx_file in glob.glob(os.path.join(run_dir, "**", "*.onnx"), recursive=True):
                if os.path.dirname(onnx_file) != onnx_dir:
                    shutil.move(onnx_file, os.path.join(onnx_dir, os.path.basename(onnx_file)))
            # keep the calib sidecar alongside the exported model too
            shutil.copy(calib, os.path.join(onnx_dir, os.path.basename(calib)))

        # --- 4) Only now mark this (config, seed) complete. ---
        shutil.copy(cfg_path, os.path.join(run_dir, "aug_config_used.json"))
        with open(done_marker, "w") as f:
            f.write(f"completed seed={seed}\n")
        print(f"[DONE] {cfg_name} seed={seed}")

print("\n>>> Grid runner finished this pass.")


Output streaming troncato alle ultime 5000 righe.
158/300: 100%|██████████| 14/14 [00:01<00:00,  7.38batch/s, avg_loss=0.0188, batch_loss=0.0667, norm=1.02]

100%|██████████| 78/78 [00:03<00:00, 23.76it/s]

159/300: 100%|██████████| 14/14 [00:09<00:00,  1.45batch/s, avg_loss=0.0246, batch_loss=0.0103, norm=0.38]
Applying Piecewise Min-Max Calibration (Target TH: 0.5)...
I-AUROC: 0.9486 AP-det: 0.9759 seg-I-AUROC: 0.8839 seg-AP-det: 0.9624 P-AUROC: 0.9503 AUPRO: 0.8763 AP-loc: 0.5262 Precision: 0.9677 Recall: 0.9677 F1-score: 0.9677 Pixel-F1: 0.2020 | Opt-Th: 0.5000

160/300: 100%|██████████| 14/14 [00:01<00:00,  7.29batch/s, avg_loss=0.0219, batch_loss=0.00096, norm=0.0962]

161/300: 100%|██████████| 14/14 [00:01<00:00,  7.48batch/s, avg_loss=0.0182, batch_loss=0.0144, norm=0.272]

162/300: 100%|██████████| 14/14 [00:01<00:00,  7.36batch/s, avg_loss=0.0201, batch_loss=0.00847, norm=0.214]

163/300: 100%|██████████| 14/14 [00:02<00:00,  6.91batch/s, avg_loss=0.0327, batch_loss=0.072, no

## 4) Summary — aggregate metrics across seeds (mean ± std)

Writes two CSVs to `ABLATION_BASE`:
- `ablation_per_seed_<CATEGORY>.csv` — one row per `(config, seed)` (the raw runs).
- `ablation_summary_<CATEGORY>.csv` — one row per config, with **mean** and
  **sample std (ddof=1)** across seeds for every metric.

`SEEDS` here must match the grid runner. Std is `0.0` when only one seed ran.

In [4]:
!pip install --upgrade --force-reinstall pandas numpy

  Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached pandas-3.0.5-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.0 MB)
Using cached numpy-2.5.2-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: python-dat

In [3]:
import os, glob, json
import numpy as np
import pandas as pd

# Define paths explicitly to avoid relative path issues
ABLATION_BASE = '/content/drive/MyDrive/Tesi/experiments/results_SSN_augmentation'
# Replace with the actual absolute path to your configs folder
CONFIG_DIR = '/content/SuperSimpleNet/configs'
CATEGORY   = 'Nero_SEMISUPERVISED_dustValidationAndTrain'
SEEDS      = [0, 1, 42]   # MUST match the seed set used by the grid runner

# Metrics we surface first (others found in metrics.json are appended automatically).
PREFERRED = ["I-AUROC", "P-AUROC", "AUPRO", "AP-loc", "AP-det",
             "F1-score", "Precision", "Recall", "seg-I-AUROC", "seg-AP-det"]

# ---- 1) Collect one row per (config, seed) ----
per_seed_rows = []
for cfg_path in sorted(glob.glob(os.path.join(CONFIG_DIR, '*.json'))):
    cfg_name = os.path.splitext(os.path.basename(cfg_path))[0]
    for seed in SEEDS:
        run_dir = os.path.join(ABLATION_BASE, f"{CATEGORY}__{cfg_name}__seed{seed}")
        metric_files = glob.glob(os.path.join(run_dir, "**", "metrics.json"), recursive=True)
        if not metric_files:
            per_seed_rows.append({"config": cfg_name, "seed": seed, "status": "not_run"})
            continue
        with open(sorted(metric_files)[0]) as f:
            m = json.load(f)
        row = {"config": cfg_name, "seed": seed, "status": "done"}
        row.update(m)
        per_seed_rows.append(row)

per_seed = pd.DataFrame(per_seed_rows)

# ---- 2) Aggregate across seeds: mean +/- std per config ----
done = per_seed[per_seed["status"] == "done"] if not per_seed.empty else per_seed

if not done.empty:
    present = [k for k in PREFERRED if k in done.columns]
    extra = [c for c in done.columns
             if c not in ("config", "seed", "status") and c not in present
             and pd.api.types.is_numeric_dtype(done[c])]
    metric_cols = present + extra
else:
    metric_cols = []

agg_rows = []
if not done.empty:
    for cfg_name, grp in done.groupby("config"):
        row = {"config": cfg_name,
               "n_seeds": int(len(grp)),
               "seeds": ",".join(map(str, sorted(grp["seed"].tolist())))}
        for k in metric_cols:
            vals = grp[k].astype(float).to_numpy()
            row[f"{k}_mean"] = float(np.mean(vals))
            row[f"{k}_std"]  = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
        agg_rows.append(row)

agg = pd.DataFrame(agg_rows)
if not agg.empty:
    ordered = ["config", "n_seeds", "seeds"]
    for k in metric_cols:
        ordered += [f"{k}_mean", f"{k}_std"]
    agg = agg[[c for c in ordered if c in agg.columns]]
    if "I-AUROC_mean" in agg.columns:
        agg = agg.sort_values("I-AUROC_mean", ascending=False)

# ---- 3) Save both tables ----
per_seed_csv = os.path.join(ABLATION_BASE, f"ablation_per_seed_{CATEGORY}.csv")
agg_csv      = os.path.join(ABLATION_BASE, f"ablation_summary_{CATEGORY}.csv")
per_seed.to_csv(per_seed_csv, index=False)
agg.to_csv(agg_csv, index=False)

# ---- 4) Pretty-print ----
print("=== PER-SEED (raw runs) ===")
print(per_seed.to_string(index=False) if not per_seed.empty else "(nothing found)")

print("\n=== AGGREGATED (mean +/- sample std across seeds) ===")
if not agg.empty:
    show = agg.copy()
    for k in metric_cols:
        mcol, scol = f"{k}_mean", f"{k}_std"
        if mcol in show.columns:
            show[k] = show.apply(lambda r: f"{r[mcol]:.4f} \u00b1 {r[scol]:.4f}", axis=1)
    display_cols = ["config", "n_seeds"] + [k for k in metric_cols if k in show.columns]
    print(show[display_cols].to_string(index=False))
else:
    print("(no completed runs yet)")

print("\nSaved per-seed ->", per_seed_csv)
print("Saved summary  ->", agg_csv)


                 config status  I-AUROC  P-AUROC    AUPRO   AP-loc   AP-det  F1-score  seg-I-AUROC  seg-AP-det  Precision   Recall  Pixel-F1
             aug_legacy   done 0.957921 0.959450 0.925447 0.641357 0.983140  0.975806     0.954214    0.985392   0.975806 0.975806  0.270023
                aug_off   done 0.950182 0.974758 0.946701 0.671666 0.972994  0.948240     0.946995    0.976584   0.974468 0.923387  0.302455
     combined_candidate   done 0.944329 0.972948 0.940909 0.633172 0.975003  0.969940     0.929761    0.975093   0.964143 0.975806  0.279840
     combined_geometric   done 0.960588 0.972711 0.951350 0.683260 0.974375  0.956522     0.957336    0.980142   0.982979 0.931452  0.305347
             oat_affine   done 0.955580 0.968932 0.944038 0.684783 0.978876  0.971774     0.949727    0.982011   0.971774 0.971774  0.309932
               oat_blur   done 0.951483 0.972448 0.948083 0.686269 0.975425  0.974052     0.952523    0.977782   0.964427 0.983871  0.301693
oat_brightnes